# FIRMS Fire Counts — NASA VIIRS/MODIS

**Nguồn:** NASA FIRMS Area API (VIIRS Suomi-NPP 375m archive)  
**Mục đích:** Đếm số điểm cháy + tổng FRP trong vùng bán kính 200km / 500km quanh mỗi trạm  
Lửa nông nghiệp Đông Nam Á + miền Nam Trung Quốc là nguồn PM2.5 vận chuyển tầm xa quan trọng cho Hà Nội  

**Đăng ký MAP_KEY miễn phí:** https://firms.modaps.eosdis.nasa.gov/api/map_key/  

**Features tạo ra:**
- `fire_count_200km` — số điểm cháy trong 200km  
- `fire_count_500km` — số điểm cháy trong 500km  
- `fire_frp_200km` — tổng Fire Radiative Power 200km (MW)  
- `fire_frp_500km` — tổng Fire Radiative Power 500km (MW)  

**Output:** `data/processed/firms_fire_daily.csv` → merge vào `daily_merged.csv`

## Cell 1 — Cài đặt

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "tqdm"])
print("Ready.")

Ready.


## Cell 2 — Cấu hình

> **Bước đầu tiên:** Đăng ký MAP_KEY tại https://firms.modaps.eosdis.nasa.gov/api/map_key/  
> Điền vào `MAP_KEY` bên dưới trước khi chạy Cell 3.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import requests, time, io
from datetime import datetime, timedelta
from tqdm import tqdm

# ── Điền MAP_KEY của bạn vào đây ─────────────────────────────────────────────
MAP_KEY = "6b47877eee2ae7f4d1826b0596194a82"
# ─────────────────────────────────────────────────────────────────────────────

ROOT     = Path("D:/Bussiness_plan/Multimodal_PM25")
RAW_DIR  = ROOT / "data/raw/firms"; RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR = ROOT / "data/processed"
OUT_CSV  = PROC_DIR / "firms_fire_daily.csv"
MERGED   = PROC_DIR / "daily_merged.csv"

STATIONS = {
    2161292: (21.0152, 105.7999),
    2161306: (21.0500, 105.7400),
    4946811: (21.0491, 105.8831),
    4946812: (21.0031, 105.7947),
    4946813: (21.0052, 105.8418),
    6123215: (20.9933, 105.9441),
}

BBOX = {"west": 98, "south": 13, "east": 117, "north": 28}

PRODUCT   = "VIIRS_SNPP_SP"
API_BASE  = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"

DATE_START = "2024-01-01"
DATE_END   = "2026-05-15"

RADIUS_KM = [200, 500]

print(f"Product  : {PRODUCT}")
print(f"BBox     : W{BBOX['west']} S{BBOX['south']} E{BBOX['east']} N{BBOX['north']}")
print(f"Period   : {DATE_START} -> {DATE_END}")
print(f"Radius   : {RADIUS_KM} km")
print(f"MAP_KEY  : {'SET ✓' if MAP_KEY != 'YOUR_MAP_KEY_HERE' else 'NOT SET ✗ — hãy điền vào trước'}")

Product  : VIIRS_SNPP_SP
BBox     : W98 S13 E117 N28
Period   : 2024-01-01 -> 2026-05-15
Radius   : [200, 500] km
MAP_KEY  : SET ✓


## Cell 3 — Hướng dẫn download thủ công từ FIRMS website

**FIRMS SP archive API yêu cầu permission đặc biệt (không tự động có với MAP_KEY thường).**  
Thay vào đó: download CSV trực tiếp từ website — nhanh hơn, không cần API.

### Bước 1 — Vào trang download:
👉 https://firms.modaps.eosdis.nasa.gov/download/

### Bước 2 — Cài đặt:
- **Data source**: VIIRS S-NPP (C2) hoặc VIIRS NOAA-20 (C2)  
- **Format**: CSV  
- **Date range**: 2024-01-01 → 2024-12-31 (làm 2 lần cho 2024 và 2025)  
- **Region**: vẽ bounding box: W=98 S=13 E=117 N=28 (bao phủ SEA + South China)  
- Hoặc chọn **Country**: Vietnam + China + Thailand + Myanmar + Laos + Cambodia

### Bước 3 — Lưu file vào:
```
data/raw/firms/firms_sea_2024.csv
data/raw/firms/firms_sea_2025.csv
```

### Bước 4 — Chạy Cell 4 để xử lý

In [3]:
## Cell 2b — Test API (chạy trước Cell 3 để kiểm tra)
import requests

area = f"{BBOX['west']},{BBOX['south']},{BBOX['east']},{BBOX['north']}"

# Thử các product + format khác nhau
test_cases = [
    f"{API_BASE}/{MAP_KEY}/VIIRS_SNPP_SP/{area}/2024-01-01/5",
    f"{API_BASE}/{MAP_KEY}/VIIRS_NOAA20_SP/{area}/2024-01-01/5",
    f"{API_BASE}/{MAP_KEY}/MODIS_SP/{area}/2024-01-01/5",
    f"{API_BASE}/{MAP_KEY}/VIIRS_SNPP_NRT/{area}/1",   # NRT: chỉ dùng được ~10 ngày gần nhất
]

for url in test_cases:
    r = requests.get(url, timeout=30)
    product = url.split("/")[7]
    status  = r.status_code
    preview = r.text[:120].replace("\n", " ")
    print(f"[{status}] {product:<20} → {preview}")

[400] VIIRS_SNPP_SP        → Invalid day range. Expects [1..5]. Invalid date format. Expects YYYY-MM-DD.
[400] VIIRS_NOAA20_SP      → Invalid day range. Expects [1..5]. Invalid date format. Expects YYYY-MM-DD.
[400] MODIS_SP             → Invalid day range. Expects [1..5]. Invalid date format. Expects YYYY-MM-DD.
[200] VIIRS_SNPP_NRT       → latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynig


In [4]:
from pathlib import Path

# Load tất cả file CSV trong data/raw/firms/
csv_files = sorted(RAW_DIR.glob("firms_sea_*.csv"))
print(f"Found {len(csv_files)} file(s): {[f.name for f in csv_files]}")
assert len(csv_files) > 0, (
    "Chưa có file! Download từ https://firms.modaps.eosdis.nasa.gov/download/ "
    "rồi lưu vào data/raw/firms/firms_sea_2024.csv (và 2025)"
)

frames = []
for f in csv_files:
    df = pd.read_csv(f)
    # Chuẩn hoá cột ngày (FIRMS CSV dùng 'acq_date')
    if "acq_date" in df.columns:
        df["acq_date"] = pd.to_datetime(df["acq_date"])
    else:
        date_col = [c for c in df.columns if "date" in c.lower()][0]
        df["acq_date"] = pd.to_datetime(df[date_col])
    frames.append(df)
    print(f"  {f.name}: {len(df):,} rows  ({df['acq_date'].min().date()} → {df['acq_date'].max().date()})")

fire_raw = pd.concat(frames, ignore_index=True)

# Lọc bounding box
fire_raw = fire_raw[
    (fire_raw["latitude"]  >= BBOX["south"]) & (fire_raw["latitude"]  <= BBOX["north"]) &
    (fire_raw["longitude"] >= BBOX["west"])  & (fire_raw["longitude"] <= BBOX["east"])
].reset_index(drop=True)

print(f"\nTotal fire detections (bbox): {len(fire_raw):,}")
print(f"Date range : {fire_raw['acq_date'].min().date()} → {fire_raw['acq_date'].max().date()}")
print(f"Columns    : {fire_raw.columns.tolist()}")

Found 0 file(s): []


AssertionError: Chưa có file! Download từ https://firms.modaps.eosdis.nasa.gov/download/ rồi lưu vào data/raw/firms/firms_sea_2024.csv (và 2025)

## Cell 4 — Tính khoảng cách Haversine + Aggregate theo bán kính

In [ ]:
def haversine_km(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2_arr)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


assert len(fire_raw) > 0, "fire_raw empty — chạy Cell 4 trước"

frp_col    = "frp" if "frp" in fire_raw.columns else None
fire_lats  = fire_raw["latitude"].values
fire_lons  = fire_raw["longitude"].values
fire_dates = fire_raw["acq_date"].values.astype("datetime64[D]")
fire_frp   = fire_raw[frp_col].values if frp_col else np.ones(len(fire_raw))

date_range = pd.date_range(DATE_START, DATE_END, freq="D")

records = []
for loc_id, (st_lat, st_lon) in STATIONS.items():
    print(f"  Station {loc_id} ...", end=" ", flush=True)
    dists = haversine_km(st_lat, st_lon, fire_lats, fire_lons)
    for d in date_range:
        d64 = np.datetime64(d, "D")
        mask_d = fire_dates == d64
        row = {"location_id": loc_id, "date": d}
        for r_km in RADIUS_KM:
            mask = mask_d & (dists <= r_km)
            row[f"fire_count_{r_km}km"] = int(mask.sum())
            row[f"fire_frp_{r_km}km"]   = float(fire_frp[mask].sum())
        records.append(row)
    print("OK")

fire_df = pd.DataFrame(records)
fire_df["date"] = pd.to_datetime(fire_df["date"])

print(f"\nShape: {fire_df.shape}")
print(fire_df[[c for c in fire_df.columns if "fire" in c]].describe().round(1))

## Cell 5 — Visualize seasonal fire pattern

In [ ]:
import matplotlib.pyplot as plt

fire_df["month"] = fire_df["date"].dt.month
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly = fire_df.groupby("month")["fire_count_500km"].mean()
monthly.plot(ax=axes[0], marker="o", color="firebrick")
axes[0].set_title("Fire count 500km theo tháng", fontsize=12)
axes[0].set_xlabel("Tháng"); axes[0].set_ylabel("Số điểm cháy / ngày")
axes[0].set_xticks(range(1, 13)); axes[0].grid(alpha=0.3)

sub = fire_df[fire_df["location_id"] == list(STATIONS.keys())[0]]
axes[1].plot(sub["date"], sub["fire_count_500km"], color="firebrick", alpha=0.7, linewidth=0.8)
axes[1].set_title("Fire count 500km time series", fontsize=12)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / "outputs/firms_fire_seasonal.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/firms_fire_seasonal.png")

## Cell 6 — Correlation Fire vs PM2.5

In [ ]:
import matplotlib.pyplot as plt

main = pd.read_csv(MERGED, parse_dates=["date"])
check = main.merge(fire_df[["location_id","date","fire_count_500km","fire_frp_500km"]],
                   on=["location_id","date"], how="left")

for col in ["fire_count_200km", "fire_count_500km", "fire_frp_200km", "fire_frp_500km"]:
    if col in fire_df.columns:
        check2 = main.merge(fire_df[["location_id","date",col]], on=["location_id","date"], how="left")
        valid  = check2[["pm25", col]].dropna()
        r = valid.corr().iloc[0, 1]
        print(f"  {col:<25} r={r:+.3f}  (n={len(valid)})")

fig, ax = plt.subplots(figsize=(6, 5))
valid = check[["pm25","fire_count_500km"]].dropna()
ax.scatter(valid["fire_count_500km"], valid["pm25"], alpha=0.3, s=8, color="firebrick")
ax.set_xlabel("Fire count 500km"); ax.set_ylabel("PM2.5 (ug/m3)")
ax.set_title("Fire count vs PM2.5", fontsize=12); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / "outputs/firms_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

## Cell 7 — Lưu CSV + Merge vào daily_merged.csv

In [ ]:
fire_df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}  shape={fire_df.shape}")

main = pd.read_csv(MERGED, parse_dates=["date"])
fire_cols = [c for c in fire_df.columns if c not in ["location_id","date","month"]]
for col in fire_cols:
    if col in main.columns:
        main.drop(columns=[col], inplace=True)

fire_df["date"] = pd.to_datetime(fire_df["date"])
merge_cols = ["location_id","date"] + fire_cols
main = main.merge(fire_df[merge_cols], on=["location_id","date"], how="left")

for col in fire_cols:
    pct = main[col].notna().mean() * 100
    print(f"  {col}: {pct:.1f}% coverage")

main.to_csv(MERGED, index=False)
print(f"\nUpdated: {MERGED}  shape={main.shape}")

## Cell 8 — Thêm vào train_3d_v2.py

```python
# ctx_feats
+ ["fire_count_500km", "fire_frp_500km"]

# tab_cols
+ ["fire_count_200km", "fire_count_500km", "fire_frp_200km", "fire_frp_500km"]
```

**Lưu ý:** Nên log-transform trước khi dùng vì phân phối fire count rất skewed:
```python
# trong load_data() của train_3d_v2.py
for col in ["fire_count_200km","fire_count_500km","fire_frp_200km","fire_frp_500km"]:
    if col in df.columns:
        df[f"{col}_log"] = np.log1p(df[col])
```

Xóa cache trước khi chạy lại:
```python
import shutil, glob, os
shutil.rmtree('outputs/final_3d_v2/cache', ignore_errors=True)
for f in glob.glob('outputs/final_3d_v2/neural_seed*.pt'): os.remove(f)
```